# 01 ANN と LIF-SNN の基礎

**E3：神経回路モデリング — ANN・SNN 比較と Spiking Manifesto の実装**

## 1. 研究目的

- **ANN（人工ニューラルネットワーク）** が、入力の**重み付き和**と活性化関数で連続値を出力するしくみを理解する。
- **LIF（Leaky Integrate-and-Fire）ニューロン** が、膜電位をためて閾値で**スパイク**を出すしくみを理解する。
- 「連続値で情報を表す ANN」と「スパイクの**有無・時刻**で情報を表す SNN」の違いを、グラフで確かめる。

> この Notebook は既存の `ann.py` と `lif_snn.py` をそのまま使います（計算の本体は `.py` にあり、Notebook は説明と可視化を担当します）。

## 2. 理論的背景

### ANN（全結合層 1 層）

$$ y = \mathrm{ReLU}(Wx + b), \qquad \mathrm{ReLU}(v) = \max(0, v) $$

| 記号 | 意味 |
| --- | --- |
| $x$ | 入力ベクトル（長さ $n$） |
| $W$ | 重み行列（$m$ 行 $n$ 列）。$W_{ij}$ は「入力 $j$ → 出力 $i$」の結合の強さ |
| $b$ | バイアス（長さ $m$）。入力が 0 でも足される値 |
| ReLU | 活性化関数。負の値を 0 にする（**非線形性**を入れる役割） |

出力 1 つにつき乗算 $n$ 回・加算 $n$ 回なので、全体で $O(nm)$ の積和演算です。

### LIF ニューロン（離散時間・オイラー法）

$$ V[t+1] = V[t] + \frac{dt}{\tau}\bigl(-(V[t]-V_\mathrm{rest}) + R\,I[t]\bigr) $$

$V[t+1] \ge V_\mathrm{th}$ になったら**スパイク**を出し、$V$ を $V_\mathrm{reset}$ に戻します。

- **膜電位** $V$：ニューロンの内部状態（電気的な「たまり具合」）
- **膜時定数** $\tau$：漏れの速さ。大きいほどゆっくり漏れる（＝ゆっくりたまる）
- **閾値** $V_\mathrm{th}$：これを超えると発火する
- $R\,I \le V_\mathrm{th}$ のとき、$V$ は $R\,I$ に近づくだけで**発火しません**

> ⚠️ LIF は「ニューロンがどう発火するか」を理解するための**導入モデル**です。
> Spiking Manifesto（論文 Section II）のモデルは、膜電位を時間ステップごとにシミュレーションせず、
> **latency vector（発火時刻のベクトル）から次の latency vector への変換を LUT で表します**（Notebook 02）。

## 3. 実験環境（最初に必ず実行）

このセルは、E3 リポジトリの `.py` ファイル（`ann.py`、`spike_lut.py` など）を import できるように、
**作業ディレクトリ（E3 のルート）を自動で探して設定**します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのチームのリポジトリの URL** を入れる（`REPO_SUBDIR` に E3 のフォルダ名） |
| Colab（方法B：ZIP をアップロード） | E3 フォルダを ZIP にして、Colab の左の「ファイル」に `/content` へアップロードしておく（またはこのセルの実行中にアップロード） |

> ⚠️ `textbook/` フォルダ（教科書のコード）は**公開リポジトリには含めない**約束です。
> 方法A で `textbook/` がない場合、`spike_encoding.py`・`benchmark.py` の import が失敗します。
> その場合は方法B（ZIP）を使うか、授業で配布された `textbook/` を同じ場所に置いてください。

In [ ]:
import os
import sys
import glob
import zipfile
import subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                           # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "e3-ann-snn-comparison"   # リポジトリの中の E3 フォルダ（ルートが E3 なら "" にする）

IN_COLAB = "google.colab" in sys.modules


def is_e3_root(p):
    """spike_lut.py と ann.py があるフォルダを E3 のルートとみなす"""
    return (Path(p) / "spike_lut.py").exists() and (Path(p) / "ann.py").exists()


def find_e3_root():
    # 1) ローカル：notebooks/ の中、または E3 のルートで開いた場合
    for p in [Path.cwd(), Path.cwd().parent]:
        if is_e3_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    # 2) Colab 方法A：clone
    if REPO_URL:
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_e3_root(cand):
            return cand
    # 3) Colab 方法B：/content にある ZIP を展開（なければアップロードを求める）
    zips = glob.glob("/content/*.zip")
    if not zips:
        from google.colab import files
        print("E3 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/e3_upload")
    for hit in glob.glob("/content/e3_upload/**/spike_lut.py", recursive=True):
        if is_e3_root(Path(hit).parent):
            return Path(hit).parent
    return None


E3_ROOT = find_e3_root()
if E3_ROOT is None:
    raise RuntimeError("E3 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(E3_ROOT)
if str(E3_ROOT) not in sys.path:
    sys.path.insert(0, str(E3_ROOT))
print("E3_ROOT:", E3_ROOT)
if not (E3_ROOT / "textbook" / "insertion_sort.py").exists():
    print("⚠️ textbook/ がありません。spike_encoding.py などの import が失敗します（上の注意を参照）。")

In [ ]:
import matplotlib.pyplot as plt

# グラフの共通スタイル（Colab には日本語フォントがないことが多いので、図の文字は英語にしている）
plt.rcParams.update({
    "figure.figsize": (7, 3.5), "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2,
})
# 色は役割ごとに固定する（ANN / LIF・SNN / LUT、CPU / GPU / TPU）
COLORS = {"ANN": "#2a78d6", "LIF": "#1baf7a", "LUT": "#eb6834",
          "CPU": "#2a78d6", "GPU": "#eb6834", "TPU": "#1baf7a", "neutral": "#8a8985"}

## 4. 使用するコード

| ファイル | 使う関数 |
| --- | --- |
| `ann.py` | `weighted_sum`（Wx + b）、`dense_forward`（ReLU まで）、`relu`、`ann_operation_counts` |
| `lif_snn.py` | `simulate_lif`（膜電位とスパイクの記録）、`firing_rate`、`first_spike_latency`、`latency_encode` |
| `data/sample_patterns.json` | 例として使う人工データ |

In [ ]:
import json
import numpy as np   # グラフ用の配列操作だけに使う（計算の本体は ann.py・lif_snn.py）

from ann import weighted_sum, dense_forward, relu, ann_operation_counts
from lif_snn import simulate_lif, firing_rate, first_spike_latency, latency_encode

with open("data/sample_patterns.json", encoding="utf-8") as f:
    data = json.load(f)
print(data["description"])

## 5. 実行例

### Part A：ANN

`data/sample_patterns.json` の例（入力 $n=3$、出力 $m=2$）で forward 計算をします。

In [ ]:
ex = data["ann_example"]
x, W, b = ex["x"], ex["W"], ex["b"]

z = weighted_sum(x, W, b)      # 活性化関数の前：Wx + b
y = dense_forward(x, W, b)     # 活性化関数の後：ReLU(Wx + b)

print("入力 x       :", x)
print("重み W       :", W)
print("バイアス b   :", b)
print("Wx + b       :", [round(v, 3) for v in z])
print("ReLU(Wx + b) :", [round(v, 3) for v in y])
print("演算回数     :", ann_operation_counts(len(x), len(W)))

# 検算：出力 0 は 0.5*0.2 - 0.2*0.5 + 0.1*0.8 + 0.1 = 0.18、出力 1 は負なので ReLU で 0
assert abs(y[0] - 0.18) < 1e-9 and y[1] == 0.0

### 6-A. 可視化：ANN の入力・出力

左が入力 $x$、中央が活性化前の $Wx+b$、右が ReLU 後の出力 $y$ です。負の値が ReLU で 0 になることを確かめてください。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].bar(range(len(x)), x, color=COLORS["neutral"])
axes[0].set_title("Input x"); axes[0].set_xlabel("input index j")
axes[1].bar(range(len(z)), z, color=COLORS["ANN"], alpha=0.5)
axes[1].axhline(0, color="black", lw=1)
axes[1].set_title("Wx + b (before ReLU)"); axes[1].set_xlabel("output index i")
axes[2].bar(range(len(y)), y, color=COLORS["ANN"])
axes[2].set_title("y = ReLU(Wx + b)"); axes[2].set_xlabel("output index i")
for ax in axes[1:]:
    ax.set_xticks(range(len(z)))
plt.tight_layout(); plt.show()

### 入力の大きさと ANN の出力（連続的な変化）

入力 $x$ 全体を $s$ 倍したときの出力を調べます。ANN の出力は入力に対して**連続的に**変わります。

In [ ]:
scales = np.linspace(-1.0, 2.0, 61)
outputs = np.array([dense_forward([s * v for v in x], W, b) for s in scales])

plt.figure()
for i in range(outputs.shape[1]):
    plt.plot(scales, outputs[:, i], label=f"output {i}",
             color=[COLORS["ANN"], COLORS["LUT"]][i % 2])
plt.xlabel("input scale s  (input = s * x)"); plt.ylabel("ANN output y_i")
plt.title("ANN output changes continuously with the input")
plt.legend(); plt.show()

### Part B：LIF-SNN

一定の入力電流 $I = 1.5$ を 50 ステップ与えます（$dt=1$、$\tau=10$、$V_\mathrm{th}=1$）。

In [ ]:
ex = data["lif_example"]
record = simulate_lif([ex["current"]] * ex["steps"], dt=ex["dt"], tau=ex["tau"], v_th=ex["v_th"])

print("スパイク時刻 :", record["spike_times"])
print("スパイク数   :", record["spike_count"])
print("発火率       :", round(firing_rate(record), 4), "spikes/step")

# 検算：既定値で I = 1.5 のとき t = 11, 22, 33, 44 に発火する（README 4章）
assert record["spike_times"] == [11.0, 22.0, 33.0, 44.0]

### 6-B. 可視化：膜電位・閾値・スパイク時刻

上段が膜電位 $V$（破線が閾値 $V_\mathrm{th}$）、下段がスパイク時刻です。
`simulate_lif` は、発火したステップでは**リセット後の値**を記録するので、グラフ上で $V$ が閾値に届いた点は見えず、0 に落ちる点として見えます。

In [ ]:
def plot_lif(record, v_th, title):
    t = record["times"]
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 4), sharex=True,
                                   gridspec_kw={"height_ratios": [3, 1]})
    ax1.plot(t, record["v"], color=COLORS["LIF"], label="membrane potential V")
    ax1.axhline(v_th, color="black", ls="--", lw=1, label="threshold V_th")
    ax1.set_ylabel("V"); ax1.set_title(title); ax1.legend(loc="lower right")
    ax2.eventplot(record["spike_times"], colors=COLORS["LIF"], lineoffsets=0.5, linelengths=0.8)
    ax2.set_yticks([]); ax2.set_ylabel("spikes"); ax2.set_xlabel("time step")
    plt.tight_layout(); plt.show()

plot_lif(record, ex["v_th"], f"LIF neuron, constant input I = {ex['current']}")

### 6-C. 可視化：入力電流と発火率・最初のスパイク時刻

入力電流 $I$ を変えると、**発火率**（一定時間内のスパイク数）と、**最初のスパイク時刻（latency）**が変わります。
$R\,I \le V_\mathrm{th}$（ここでは $I \le 1$）では発火しません。

In [ ]:
steps = 200
currents = np.round(np.arange(0.5, 4.01, 0.05), 2)
rates, latencies = [], []
for current in currents:
    rec = simulate_lif([float(current)] * steps)
    rates.append(firing_rate(rec))
    latencies.append(rec["spike_times"][0] if rec["spike_times"] else np.nan)  # 発火しなければ NaN

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ax1.plot(currents, rates, color=COLORS["LIF"])
ax1.axvline(1.0, color=COLORS["neutral"], ls=":", lw=1)
ax1.set_xlabel("input current I"); ax1.set_ylabel("firing rate [spikes/step]")
ax1.set_title("f-I curve (rate code)")
ax2.plot(currents, latencies, color=COLORS["LUT"])
ax2.axvline(1.0, color=COLORS["neutral"], ls=":", lw=1)
ax2.set_xlabel("input current I"); ax2.set_ylabel("first spike time")
ax2.set_title("latency code: stronger input fires earlier")
plt.tight_layout(); plt.show()

print("latency_encode([1.2, 1.5, 2.0, 3.0]) =", latency_encode([1.2, 1.5, 2.0, 3.0]))
assert latency_encode([1.2, 1.5, 2.0, 3.0]) == [18.0, 11.0, 7.0, 4.0]

## 7. 結果の解釈

- **ANN**：出力は連続値です。入力を少し変えると出力も少し変わります。ReLU は負の値を 0 にし、「反応しない出力」を作ります。
- **LIF**：出力は「スパイクが出たか・出ないか」という**離散的な事象**です。入力の強さは、
  - **発火率**（スパイクの数）としても、
  - **最初のスパイク時刻**（強いほど早い ＝ latency）としても
  表すことができます。
- Spiking Manifesto が重視するのは後者、つまり**スパイクの時刻・順序**です。これを Notebook 02 で扱います。
- ここでの LIF は 1 個のニューロンを時間ステップごとに計算するもので、論文の LUT モデルとは**計算のしかたが違う**ことに注意してください。

## 8. 学生向け TODO

下のセルの値を変えて、もう一度実行してください。結果は表にまとめましょう。

- [ ] **TODO 1**：ANN の重み `W_new` を変更し、出力がどう変わるか調べる。
- [ ] **TODO 2**：ReLU を使わない場合（`activation=lambda v: v`）と比べ、ReLU の効果を調べる。
- [ ] **TODO 3**：LIF の閾値 `v_th` を変更する。
- [ ] **TODO 4**：入力電流 `current` を変更する（1.0 付近を細かく）。
- [ ] **TODO 5**：膜時定数 `tau` を変更する。
- [ ] **TODO 6**：条件ごとのスパイク発生回数を表にして比較する。

In [ ]:
# ---- TODO 1・2：ANN ----
W_new = [[0.5, -0.2, 0.1], [-0.3, 0.8, -0.6]]   # TODO：値を変えてみよう
print("ReLU あり:", dense_forward(x, W_new, b))
print("ReLU なし:", dense_forward(x, W_new, b, activation=lambda v: v))

# ---- TODO 3〜6：LIF ----
current, tau, v_th, steps = 1.5, 10.0, 1.0, 100   # TODO：値を変えてみよう
rec = simulate_lif([current] * steps, tau=tau, v_th=v_th)
print(f"I={current}, tau={tau}, v_th={v_th}: spikes={rec['spike_count']}, "
      f"first={rec['spike_times'][:1]}, rate={firing_rate(rec):.3f}")
# TODO 6：for ループで条件を変え、結果を表（リストや辞書）にまとめよう

## 9. 考察課題

1. $R\,I \le V_\mathrm{th}$ のとき LIF が発火しない理由を、式から説明してください。
2. ANN の出力（連続値）と LIF の出力（スパイク）では、**同じ入力の強さ**がどのように違う形で表されていますか。
3. 「発火率」で情報を表す場合と、「最初のスパイク時刻」で表す場合では、情報を読み取るのに必要な**観測時間**はどう違いますか。
4. LIF のシミュレーションでは時間ステップの数だけ計算が必要です。ANN の forward 1 回と比べて、計算量はどう違いますか（Notebook 03 で測定します）。

## 10. 参考文献

- E. Izhikevich, "Spiking Manifesto", arXiv:2512.11843, 2025. <https://arxiv.org/abs/2512.11843>（arXiv の表示ライセンス：CC BY-NC-SA 4.0）
- 著者の公開コード：<https://github.com/izhikevich/SNN>（この教材には**コピーしていません**）
- W. Gerstner, W. M. Kistler, R. Naud, L. Paninski, *Neuronal Dynamics*, Cambridge University Press, 2014（LIF モデル）
- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第3章 探索、第5章 再帰、第6章 ソート）
- E3 リポジトリの `README.md`（特に 17章「論文との対応関係」）

> この Notebook は論文の中心的なアイデアを**教育用に簡略化**したものです。論文のモデル・著者の実装そのものではありません。